# Semana 4: Regresión Logística y Métricas de Clasificación
## Notebook Conceptual (NB1) – Datos Dummy y Experimentación

**Propósito:** Entender la regresión logística como modelo de clasificación, la función sigmoide, la log-loss, y aprender a evaluar clasificadores con métricas adecuadas.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Visualizar la función sigmoide y su derivada.
- Implementar la log-loss y su gradiente desde cero.
- Entrenar regresión logística con scikit-learn y experimentar con regularización.
- Calcular métricas de clasificación (matriz de confusión, precisión, recall, F1).
- Dibujar la curva ROC y calcular AUC.
- Experimentar con clases desbalanceadas y técnicas de balanceo.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.datasets import make_classification

# Para SMOTE
from imblearn.over_sampling import SMOTE

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Función Sigmoide y su Derivada

La función sigmoide (o logística) es la base de la regresión logística:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Su derivada tiene una forma elegante:

$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

Visualizamos ambas.

In [ ]:
# Definimos la función sigmoide y su derivada
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

# Creamos un rango de valores
z = np.linspace(-10, 10, 200)
sigma = sigmoid(z)
sigma_prime = sigmoid_derivative(z)

# Visualización
plt.figure(figsize=(10, 5))
plt.plot(z, sigma, 'b-', linewidth=2, label='σ(z)')
plt.plot(z, sigma_prime, 'r-', linewidth=2, label="σ'(z)")
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.xlabel('z')
plt.ylabel('Valor')
plt.title('Función Sigmoide y su Derivada')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Implementación de Log-Loss y su Gradiente

La función de pérdida para regresión logística (entropía cruzada o log-loss) para una muestra es:

$$\mathcal{L}(\hat{p}, y) = -[y \log(\hat{p}) + (1-y) \log(1-\hat{p})]$$

donde $\hat{p} = \sigma(\beta^T x)$.

El gradiente con respecto a los coeficientes es:

$$\nabla J(\beta) = \frac{1}{n} \sum_{i=1}^n (\hat{p}_i - y_i) x_i$$

Implementamos ambas funciones y probamos con un ejemplo pequeño.

In [ ]:
# Implementación de log-loss
def log_loss(y_true, y_pred_proba, eps=1e-15):
    """
    y_true: valores reales (0 o 1)
    y_pred_proba: probabilidades predichas (entre 0 y 1)
    """
    # Clip para evitar log(0)
    y_pred_proba = np.clip(y_pred_proba, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred_proba) + (1 - y_true) * np.log(1 - y_pred_proba))

# Gradiente de log-loss
def log_loss_gradient(X, y_true, y_pred_proba):
    """
    X: matriz de diseño (incluyendo columna de unos para intercepto)
    y_true: valores reales
    y_pred_proba: probabilidades predichas
    """
    n = len(y_true)
    return (1/n) * X.T @ (y_pred_proba - y_true)

# Ejemplo pequeño
X_small = np.array([[1, 2], [1, 3], [1, 4], [1, 5]])  # con intercepto
y_small = np.array([0, 0, 1, 1])
beta_small = np.array([0.1, 0.2])

# Calculamos probabilidades
z_small = X_small @ beta_small
p_small = sigmoid(z_small)

loss = log_loss(y_small, p_small)
grad = log_loss_gradient(X_small, y_small, p_small)

print(f"Ejemplo con 4 muestras:")
print(f"Probabilidades predichas: {p_small.round(3)}")
print(f"Log-loss: {loss:.4f}")
print(f"Gradiente: {grad.round(4)}")

---
## 3. Generación de Datos Sintéticos para Clasificación Binaria

Creamos dos tipos de datasets:
1. **Linealmente separable** (para ver fronteras claras)
2. **No linealmente separable** (para ver limitaciones)

In [ ]:
# Datos linealmente separables
X_lin, y_lin = make_classification(
    n_samples=300, n_features=2, n_redundant=0, n_clusters_per_class=1,
    class_sep=2.0, random_state=42
)

# Datos no linealmente separables (anillos concéntricos)
from sklearn.datasets import make_moons
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

# Visualizamos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_lin[y_lin==0, 0], X_lin[y_lin==0, 1], c='blue', label='Clase 0', alpha=0.6)
axes[0].scatter(X_lin[y_lin==1, 0], X_lin[y_lin==1, 1], c='red', label='Clase 1', alpha=0.6)
axes[0].set_title('Datos Linealmente Separables')
axes[0].legend()
axes[0].grid(True)

axes[1].scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], c='blue', label='Clase 0', alpha=0.6)
axes[1].scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], c='red', label='Clase 1', alpha=0.6)
axes[1].set_title('Datos No Linealmente Separables (Moons)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 4. Regresión Logística con scikit-learn: Efecto de la Regularización

Entrenamos modelos con diferentes valores de **C** (inverso de la regularización).

- C alto → poca regularización (modelo complejo, puede sobreajustar)
- C bajo → mucha regularización (modelo más simple, mayor sesgo)

Observamos cómo cambia la frontera de decisión.

In [ ]:
# Función para dibujar la frontera de decisión
def plot_decision_boundary(X, y, model, ax, title):
    # Crear malla
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predecir sobre la malla
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Dibujar contorno y datos
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', label='Clase 0', alpha=0.6, edgecolors='k')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red', label='Clase 1', alpha=0.6, edgecolors='k')
    ax.set_title(title)
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')
    ax.legend()

# Probamos diferentes valores de C
C_values = [0.001, 0.1, 1, 100]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for i, C in enumerate(C_values):
    model = LogisticRegression(C=C, solver='liblinear', random_state=42)
    model.fit(X_lin, y_lin)
    plot_decision_boundary(X_lin, y_lin, model, axes[i], f'C = {C}')

plt.tight_layout()
plt.show()

---
## 5. Matriz de Confusión y Métricas de Clasificación

Entrenamos un modelo en los datos lineales y calculamos todas las métricas, primero manualmente y luego con scikit-learn.

In [ ]:
# Dividimos en train/test
X_train, X_test, y_train, y_test = train_test_split(X_lin, y_lin, test_size=0.3, random_state=42)

# Entrenamos modelo
model = LogisticRegression(C=1, solver='liblinear', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Matriz de confusión manual
def confusion_matrix_manual(y_true, y_pred):
    VP = np.sum((y_true == 1) & (y_pred == 1))
    VN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[VN, FP], [FN, VP]])

cm_manual = confusion_matrix_manual(y_test, y_pred)
cm_sklearn = confusion_matrix(y_test, y_pred)

print("Matriz de confusión (manual):")
print(cm_manual)
print("\nMatriz de confusión (sklearn):")
print(cm_sklearn)

# Visualización
plt.figure(figsize=(6, 5))
sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.show()

In [ ]:
# Cálculo manual de métricas
def metricas_manuales(cm):
    VN, FP, FN, VP = cm.ravel()  # cuidado: ravel ordena [VN, FP, FN, VP]??
    # En la matriz [[VN, FP], [FN, VP]]
    VN, FP = cm[0, 0], cm[0, 1]
    FN, VP = cm[1, 0], cm[1, 1]
    
    accuracy = (VP + VN) / (VP + VN + FP + FN)
    precision = VP / (VP + FP) if (VP + FP) > 0 else 0
    recall = VP / (VP + FN) if (VP + FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return accuracy, precision, recall, f1

acc_manual, prec_manual, rec_manual, f1_manual = metricas_manuales(cm_sklearn)

print("=== Métricas (manual) ===")
print(f"Accuracy: {acc_manual:.4f}")
print(f"Precision: {prec_manual:.4f}")
print(f"Recall: {rec_manual:.4f}")
print(f"F1-score: {f1_manual:.4f}")

print("\n=== Métricas (sklearn) ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

---
## 6. Curva ROC y AUC

La curva ROC muestra el rendimiento del clasificador para todos los umbrales posibles. El AUC (área bajo la curva) es una métrica global independiente del umbral.

In [ ]:
# Calculamos FPR, TPR para diferentes umbrales
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

# Visualización
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Aleatorio (AUC = 0.5)')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.title('Curva ROC')
plt.legend()
plt.grid(True)
plt.show()

# Mostrar algunos umbrales
for i, thresh in enumerate(thresholds[::10]):
    print(f"Umbral: {thresh:.3f} -> FPR: {fpr[i*10]:.3f}, TPR: {tpr[i*10]:.3f}")

---
## 7. Experimentación con Clases Desbalanceadas

Creamos un dataset intencionalmente desbalanceado (clase minoritaria ~10%) y comparamos:
1. Regresión logística sin ajustes
2. Con pesos de clase (class_weight='balanced')
3. Con sobremuestreo SMOTE

In [ ]:
# Generamos datos desbalanceados
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=2, n_redundant=0, n_clusters_per_class=1,
    weights=[0.9, 0.1], flip_y=0, random_state=42
)

print(f"Distribución de clases:")
print(pd.Series(y_imb).value_counts())
print(f"\nProporción clase 1: {np.mean(y_imb):.2%}")

# Visualización
plt.figure(figsize=(8, 5))
plt.scatter(X_imb[y_imb==0, 0], X_imb[y_imb==0, 1], c='blue', label='Clase 0 (90%)', alpha=0.6)
plt.scatter(X_imb[y_imb==1, 0], X_imb[y_imb==1, 1], c='red', label='Clase 1 (10%)', alpha=0.6)
plt.title('Dataset con Clases Desbalanceadas')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Dividimos en train/test
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb
)

# 1. Modelo sin ajustes
lr_simple = LogisticRegression(C=1, solver='liblinear', random_state=42)
lr_simple.fit(X_train_imb, y_train_imb)
y_pred_simple = lr_simple.predict(X_test_imb)

# 2. Modelo con pesos de clase
lr_balanced = LogisticRegression(C=1, class_weight='balanced', solver='liblinear', random_state=42)
lr_balanced.fit(X_train_imb, y_train_imb)
y_pred_balanced = lr_balanced.predict(X_test_imb)

# 3. Modelo con SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_imb, y_train_imb)

lr_smote = LogisticRegression(C=1, solver='liblinear', random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = lr_smote.predict(X_test_imb)

In [ ]:
# Evaluamos con métricas para clase minoritaria (clase 1)
def reporte_modelo(y_true, y_pred, nombre):
    precision = precision_score(y_true, y_pred, pos_label=1)
    recall = recall_score(y_true, y_pred, pos_label=1)
    f1 = f1_score(y_true, y_pred, pos_label=1)
    accuracy = accuracy_score(y_true, y_pred)
    return pd.DataFrame({
        'Modelo': [nombre],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1': [f1]
    })

df_simple = reporte_modelo(y_test_imb, y_pred_simple, 'Sin ajuste')
df_balanced = reporte_modelo(y_test_imb, y_pred_balanced, 'Class Weight')
df_smote = reporte_modelo(y_test_imb, y_pred_smote, 'SMOTE')

resultados = pd.concat([df_simple, df_balanced, df_smote], ignore_index=True)
print("=== Comparación en Clases Desbalanceadas ===")
resultados.round(4)

In [ ]:
# Visualización de fronteras de decisión
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_decision_boundary(X_test_imb, y_test_imb, lr_simple, axes[0], 'Sin ajuste')
plot_decision_boundary(X_test_imb, y_test_imb, lr_balanced, axes[1], 'Class Weight')
plot_decision_boundary(X_test_imb, y_test_imb, lr_smote, axes[2], 'SMOTE')

plt.tight_layout()
plt.show()

---
## 8. Experimentación Adicional: Efecto del Umbral de Decisión

En lugar de usar el umbral por defecto (0.5), podemos ajustarlo para favorecer la detección de la clase minoritaria.

In [ ]:
# Obtenemos probabilidades del modelo sin ajuste
y_proba_simple = lr_simple.predict_proba(X_test_imb)[:, 1]

# Probamos diferentes umbrales
umbrales = [0.3, 0.4, 0.5, 0.6, 0.7]
resultados_umbral = []

for umbral in umbrales:
    y_pred_umbral = (y_proba_simple >= umbral).astype(int)
    precision = precision_score(y_test_imb, y_pred_umbral, pos_label=1, zero_division=0)
    recall = recall_score(y_test_imb, y_pred_umbral, pos_label=1, zero_division=0)
    f1 = f1_score(y_test_imb, y_pred_umbral, pos_label=1, zero_division=0)
    resultados_umbral.append({
        'Umbral': umbral,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })

df_umbral = pd.DataFrame(resultados_umbral)
print("=== Efecto del Umbral de Decisión (modelo sin ajuste) ===")
df_umbral.round(4)

---
## 9. Conclusiones

Hemos explorado los fundamentos de la regresión logística y la evaluación de clasificadores:

✔️ **Función sigmoide**: Transforma combinaciones lineales en probabilidades.
✔️ **Log-loss y gradiente**: Base del entrenamiento.
✔️ **Regularización (C)**: Controla la complejidad del modelo.
✔️ **Matriz de confusión y métricas**: Precisión, recall, F1 según el contexto.
✔️ **Curva ROC y AUC**: Evaluación independiente del umbral.
✔️ **Clases desbalanceadas**: Técnicas como pesos de clase y SMOTE mejoran la detección de la clase minoritaria.

**Lección clave**: La elección de la métrica debe alinearse con el objetivo de negocio. En problemas desbalanceados, la exactitud (accuracy) es engañosa; debemos centrarnos en precisión, recall, F1 y AUC.

En el próximo notebook (NB2) aplicaremos estos conceptos a un problema real de detección de spam.

---
**Fin del Notebook Conceptual - Semana 4**